# Installation of Dependencies


In [1]:
!pip -q install transformers faiss-cpu langchain_huggingface langchain_elasticsearch torch sentence-transformers chromadb langchain_groq

In [1]:
# Create requirements.txt file with all installed packages
!pip freeze > requirements.txt
print("requirements.txt created successfully!")

# View the contents
!head -20 requirements.txt

requirements.txt created successfully!
annotated-types==0.7.0
anyio==4.11.0
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
arrow==1.4.0
asttokens==3.0.0
async-lru==2.0.5
attrs==25.4.0
babel==2.17.0
backoff==2.2.1
bcrypt==5.0.0
beautifulsoup4==4.14.2
bleach==6.3.0
build==1.3.0
cachetools==6.2.2
certifi==2025.10.5
cffi==2.0.0
charset-normalizer==3.4.4
chromadb==1.3.5
click==8.3.1


# Importing Libraries

In [2]:
import faiss
import numpy as np
from elasticsearch import Elasticsearch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BertTokenizer, BertForSequenceClassification
from sentence_transformers import SentenceTransformer
import chromadb
import os
import json

# Load configuration from .config file
def load_config(config_path='.config'):
    """Load API keys from config file."""
    with open(config_path, 'r') as f:
        return json.load(f)

# Load configuration
config = load_config()


/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#Load Models and Tokenizers

In [3]:
from huggingface_hub import login

# Your Hugging Face API token (You can find it in your Hugging Face account settings)
hf_api_token = config.get('HUGGING_FACE_API_KEY')


# Log in to Hugging Face
login(token=hf_api_token)


In [4]:
# Load the pre-trained models and tokenizers for text generation, sentence embedding,
# and reranking.

# Load the SentenceTransformer model for encoding queries and documents
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')  # Small, fast model for embeddings

In [3]:
# Reload config to ensure we have the latest values
config = load_config()

# Verify the API key is loaded correctly
#print(f"GROQ_API_KEY loaded: {config.get('GROQ_API_KEY')[:20]}...")  # Show first 20 chars only

# Check what models are available
from groq import Groq
client = Groq(api_key=config.get('GROQ_API_KEY'))
models = client.models.list()
for model in models.data:
    print(model.id)


openai/gpt-oss-20b
moonshotai/kimi-k2-instruct
playai-tts
allam-2-7b
qwen/qwen3-32b
llama-3.1-8b-instant
meta-llama/llama-prompt-guard-2-22m
meta-llama/llama-guard-4-12b
whisper-large-v3-turbo
playai-tts-arabic
moonshotai/kimi-k2-instruct-0905
meta-llama/llama-4-maverick-17b-128e-instruct
llama-3.3-70b-versatile
whisper-large-v3
groq/compound-mini
groq/compound
openai/gpt-oss-120b
openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-86m
meta-llama/llama-4-scout-17b-16e-instruct


In [6]:
from langchain_groq import ChatGroq

#Get your Groq AI API key from https://console.groq.com/
os.environ["GROQ_API_KEY"] = config.get('GROQ_API_KEY')

# Helper function to get the language model
def get_llm():
    """
    Returns the language model instance.

    This function initializes and returns a ChatGroq language model configured with the specified model name,
    temperature, maximum tokens, and other settings.

    Returns:
        ChatGroq: An instance of the ChatGroq language model.
    """
    llm = ChatGroq(
        model="meta-llama/llama-4-maverick-17b-128e-instruct",#"openai/gpt-oss-120b",#"mixtral-8x7b-32768",
        temperature=0,
        max_tokens=1024,
    )
    return llm


In [7]:
get_llm().invoke("Hi")

AIMessage(content="It's nice to meet you. Is there something I can help you with, or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 11, 'total_tokens': 34, 'completion_time': 0.022120389, 'prompt_time': 0.933720996, 'queue_time': 1.526367955, 'total_time': 0.955841385}, 'model_name': 'meta-llama/llama-4-maverick-17b-128e-instruct', 'system_fingerprint': 'fp_9b0c2006ef', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--009aebe9-f576-49cf-ae57-8cb33feb434f-0', usage_metadata={'input_tokens': 11, 'output_tokens': 23, 'total_tokens': 34})

# Setup Elasticsearch for Textual Retrieval
# Elasticsearch is used here to perform text-based search on indexed documents.


In [8]:
# Initialize Hugging Face Embeddings
embeddings = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")


In [9]:
from elasticsearch import Elasticsearch, ConnectionError
from langchain_elasticsearch import ElasticsearchStore

In [10]:
from elasticsearch import Elasticsearch

# Load Elasticsearch credentials from config
es_host = config.get('ELASTICSEARCH_HOST')
es_api_key = config.get('ELASTICSEARCH_PASSWORD')  # This is actually the API key

# Initialize Elasticsearch client with API key
es = Elasticsearch(
    es_host,
    api_key=es_api_key
)

# Test connection
es.ping()


True

In [11]:
# Define the index name
index_name = 'movies'

# Check if index exists, if not create it
if not es.indices.exists(index=index_name):
    # Create index with mapping (updated syntax - no 'body' parameter)
    es.indices.create(
        index=index_name,
        mappings={
            "properties": {
                "content": {"type": "text"}
            }
        }
    )
    print(f"Index '{index_name}' created.")
else:
    print(f"Index '{index_name}' already exists.")


Index 'movies' already exists.


# Load Reranking Model
# Load a pre-trained BERT model and tokenizer for reranking retrieved documents.


In [12]:
# Load the tokenizer for the reranking model
rerank_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Load the BERT model for sequence classification (used for reranking)
rerank_model = BertForSequenceClassification.from_pretrained('bert-base-uncased')


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
# Get the ChatGroq language model
chat_groq_model = get_llm()

# Function for Advanced Query Transformation
# This function modifies and expands the input query for better retrieval results.


In [14]:
def advanced_query_transformation(query):
    """
    Transforms the input query by adding synonyms, extensions, or modifying the structure
    for better search performance.

    Args:
        query (str): The original query.

    Returns:
        str: The transformed query with added synonyms or related terms.
    """
    # Example transformation: adding an OR clause with a related term
    expanded_query = query + " OR related_term"
    return expanded_query

# Function for Advanced Query Routing
# This function decides the retrieval method (textual or vector-based) based on the query type.


In [15]:
def advanced_query_routing(query):
    """
    Determines the retrieval method based on the presence of specific keywords in the query.

    Args:
        query (str): The user's query.

    Returns:
        str: 'textual' if the query requires text-based retrieval, 'vector' otherwise.
    """
    if "specific_keyword" in query:
        return "textual"
    else:
        return "vector"

# Fusion Retrieval Function
# This function retrieves documents using both vector-based and textual retrieval methods.


In [ ]:
# Fusion Retrieval Function
def fusion_retrieval(query, top_k=5):
    """
    Retrieves the top_k most relevant documents using a combination of vector-based
    and textual retrieval methods.

    Args:
        query (str): The search query.
        top_k (int): The number of top documents to retrieve.

    Returns:
        list: A list of combined results from both vector and textual retrieval methods.
    """
    # Vector-based retrieval using sentence embeddings
    query_embedding = sentence_model.encode(query).tolist()
    vector_results = collection.query(query_embeddings=[query_embedding], n_results=min(top_k, len(documents)))

    # Textual retrieval using Elasticsearch (updated syntax without 'body' parameter)
    es_results = es.search(
        index=index_name,
        size=top_k,
        query={
            "match": {
                "content": query
            }
        }
    )
    es_documents = [hit["_source"]["content"] for hit in es_results['hits']['hits']]

    # Combine results from both retrieval methods
    combined_results = vector_results['documents'][0] + es_documents

    return combined_results


# Document Reranking Function
# This function reranks retrieved documents based on their relevance to the query.


In [20]:
import torch.nn.functional as F

# Document Reranking Function
def rerank_documents(query, documents):
    """
    Reranks the retrieved documents based on their relevance to the query using a pre-trained
    BERT model.

    Args:
        query (str): The user's query.
        documents (list): A list of documents retrieved from the search.

    Returns:
        list: A list of reranked documents, sorted by relevance.
    """
    inputs = [rerank_tokenizer.encode_plus(query, doc, return_tensors='pt', truncation=True, padding=True) for doc in documents]

    # Use logits to get scores
    scores = []
    for input in inputs:
        outputs = rerank_model(**input)
        logits = outputs.logits
        probabilities = F.softmax(logits, dim=1)
        positive_class_probability = probabilities[:, 1].item()  # Assuming the second element represents the positive class
        scores.append(positive_class_probability)

    ranked_docs = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, score in ranked_docs]


# Context Selection and Compression
# This function selects and summarizes the context to be used in the final answer generation.


In [21]:
# Load summarization model
summarizer = pipeline("summarization")

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


In [22]:
def select_and_compress_context(documents):
    """
    Summarizes the content of the retrieved documents to create a compressed context.

    Args:
        documents (list): A list of documents to summarize.

    Returns:
        list: A list of summarized texts for each document.
    """
    summarized_context = []
    for doc in documents:
        input_length = len(doc.split())  # Calculate input length based on word count
        max_length = min(100, input_length)  # Set max_length to input_length if smaller than 100
        summary = summarizer(doc, max_length=max_length, min_length=5, do_sample=False)[0]['summary_text']
        summarized_context.append(summary)
    return summarized_context

# Answer Generation Function
# This function generates the final answer based on the provided context and query.


In [23]:
# Answer Generation Function
def generate_answer(query, chunks, llm):
    """
    Generates an answer based on the input query and context chunks using a language model.

    Args:
        query (str): The user's query.
        chunks (list): A list of context chunks to inform the answer.
        llm (ChatGroq): An instance of the ChatGroq language model.

    Returns:
        str: The generated answer.
    """
    # Combine chunks into a single context string
    context = "\n\n".join(chunks)

    # Construct the prompt for the language model as a string
    prompt = f"""[INST]
Instruction: You're an expert in movie suggestions. Your task is to analyze carefully the context and come up with an exhaustive answer to the following question:
{query}

Here is the context to help you:

{context}

[/INST]"""

    # Invoke the language model with the prompt
    response = llm.invoke(prompt)  # Pass the prompt as a string directly

    # Since response is likely an AIMessage object, access the content directly
    generated_text = response.content

    return generated_text


# Full Advanced Retrieval-Augmented Generation (RAG) Pipeline
# This function orchestrates the entire process: query transformation, retrieval, reranking,
# context compression, and answer generation.


In [24]:
def advanced_rag_pipeline(query):
    """
    The main pipeline function for the Advanced Retrieval-Augmented Generation (RAG) system.
    It processes the query, retrieves relevant documents, reranks them, selects and compresses
    the context, and finally generates an answer.

    Args:
        query (str): The user's input query.

    Returns:
        str: The final generated answer.
    """
    # Transform and route query
    transformed_query = advanced_query_transformation(query)
    retrieval_method = advanced_query_routing(transformed_query)

    # Retrieve documents using fusion retrieval
    retrieved_documents = fusion_retrieval(transformed_query)

    # Rerank documents based on relevance
    ranked_documents = rerank_documents(query, retrieved_documents)

    # Select and compress context for answer generation
    context = select_and_compress_context(ranked_documents)

    # Generate final answer based on the context
    final_answer = generate_answer(query, context, chat_groq_model)
    return final_answer

# Example Usage of the Advanced RAG Pipeline
# Demonstrate the use of the pipeline with an example query.


In [25]:
import chromadb

# Initialize ChromaDB client and create collection
client = chromadb.Client()

# Define the collection name
collection_name = "movies"

try:
    # Attempt to create the collection in ChromaDB
    collection = client.create_collection(name=collection_name)
    print(f"Collection '{collection_name}' created successfully.")

    # Define the documents to be inserted into the collection
    documents = [
        {"id": "1", "content": "The Shawshank Redemption is a great movie to watch on a rainy day."},
        {"id": "2", "content": "Forrest Gump is an uplifting film perfect for a rainy afternoon."}
    ]

    # Extract the IDs and content for insertion
    ids = [doc["id"] for doc in documents]
    contents = [doc["content"] for doc in documents]

    # Insert documents into the collection
    collection.add(ids=ids, documents=contents)
    print("Documents inserted successfully.")

except Exception as e:
    print(f"Collection '{collection_name}' already exists. No need to create it again.")
    # Optionally, you could fetch the existing collection here
    collection = client.get_collection(name=collection_name)

except Exception as e:
    print(f"An error occurred: {e}")


Collection 'movies' created successfully.


/home/codespace/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 104MiB/s]


Documents inserted successfully.


In [41]:
# Example query
query = "What are some good movies to watch on a rainy day?"

# Run the query through the Advanced RAG Pipeline
answer = advanced_rag_pipeline(query)

# Output the generated answer
print(answer)

Based on the context, it appears that you're looking for movie suggestions that are uplifting, great, or otherwise engaging to watch on a rainy day. Given that Forrest Gump is described as an uplifting film and The Shawshank Redemption is considered a great movie, I'll suggest a list of films that share similar qualities or are otherwise suitable for a cozy rainy day.

Here are some movie suggestions for a rainy day:

1. **Uplifting Films:**
   - Forrest Gump (as mentioned) - A heartwarming story of resilience and kindness.
   - The Pursuit of Happyness - A true story of overcoming adversity.
   - Hidden Figures - An inspiring tale of women who broke barriers in STEM.
   - La La Land - A modern romantic musical with an uplifting message.

2. **Dramas and Inspirational Stories:**
   - The Shawshank Redemption (as mentioned) - A powerful story of hope and redemption.
   - The Green Mile - A touching story of compassion and justice.
   - The Notebook - A classic romance that stands the te